# Burst Detection Implementation

This netebook contains implementation of 2 burst detection methods: CUSUM change-point detection on pressure time-series, and customer contact spatial/temporal clustering for areas without sensors. The final outputs are 2 alert feeds generated by each of the detection methods to show potential incidents, their severity, advisory radius and number of pipes affected. You will also get 2 validaion results report showing false positives, false negatives, precision and recall for each of these two methods against the provided validation set.

### Instructions
- Ensure you have the following datasets in the same path as your notebook:
    - pipes.csv
    - pressure_sites.csv
    - pressure_readings.csv
    - complaints.csv
    - work_orders.csv
    - event_alerts.csv
    - event_validation.csv


- Run the notebook from top to bottom

### 1. Import required packages


In [1]:
import warnings

import pandas as pd
import numpy as np

from shapely import wkt
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import Point


from sklearn.cluster import DBSCAN

warnings.filterwarnings("ignore")


### 2. Load data

In [2]:

pressure_readings= pd.read_csv('pressure_readings.csv')
pressure_sites= pd.read_csv('pressure_sites.csv')
pipes = pd.read_csv('pipes.csv')
complaints= pd.read_csv('complaints.csv')
work_orders = pd.read_csv('work_orders.csv')


#validation set
event_alerts=pd.read_csv('event_alerts.csv')
event_affected_mains=pd.read_csv('event_affected_mains.csv')

### 3. Clean timestamp and geometries columns

In [3]:
pressure_readings["dt_stamp"] = pd.to_datetime(pressure_readings["dt_stamp"])
complaints["recorded_at"] = pd.to_datetime(complaints["recorded_at"])
event_alerts["from_date"] = pd.to_datetime(event_alerts["from_date"])
event_alerts["to_date"] = pd.to_datetime(event_alerts["to_date"])

#convert WKT points into geometry points
pressure_sites["geometry"] = pressure_sites["geom_wkt"].apply(wkt.loads)
pipes["geometry"] = pipes["geom_wkt"].apply(wkt.loads)
complaints['geometry'] = complaints['geom_wkt'].apply(wkt.loads)

#get coordinates to be used in dbscan
complaints["longitude"] = complaints["geometry"].apply(lambda p: p.x)
complaints["latitude"] = complaints["geometry"].apply(lambda p: p.y)

#interepret the geometry points as longitude/latitude
pressure_sites_gdf = gpd.GeoDataFrame(pressure_sites, geometry="geometry", crs="EPSG:4326")
pipes_gdf = gpd.GeoDataFrame(pipes,geometry="geometry", crs="EPSG:4326")
complaints_gdf = gpd.GeoDataFrame(complaints, geometry="geometry", crs="EPSG:4326")

#transform the longitude/latitude degrees into a meter-based system as it is a more interpretable unit for understanding proximity
pipes_m = pipes_gdf.to_crs("EPSG:32616")
complaints_m = complaints_gdf.to_crs("EPSG:32616")

pipes_m["id"] = pipes_m["id"].astype(str)
work_orders['pipe_id'] = work_orders['description'].str.extract(r'pipe #(\d+)')


### 4. CUSUM pressure anomaly detection

In this method of anomaly detection, I will observe pressure readings at each site and compare each reading to a rolling baseline. Since the pressure readings are available every 15mins, I will use a baseline of 96 readings (each hour has 4 readings, and there's 24 hours in a day). I will also get the standard deviation for each site, set slack (k) to 0.5 times the standard deviation and the threshold to 5 times the standard deviation to help identify the change events. To avoid false positives, I'll require the change events to have been present in at least 2 consecutive readings (30 minutes) before classifying them as abnormal. This is partly inspired by the note that Sand's Burst Detection can cut the response time to under 30mins, but mostly because a single deviation in 15mins may be inconsequential and shouldn't have to trigger an alert.

In [4]:
def get_cusum_per_site(pressure_site, rolling_window=96, k_multiplier=0.5, threshold_multiplier=5, persistence=2):
    
    #sort the data by timestamp
    pressure_site = pressure_site.sort_values("dt_stamp").copy()

    #get the recent avg reading for the site based on 96 readings
    pressure_site["rolling_mean"] = pressure_site["value"].rolling(rolling_window).mean()

    #get the pressure standard deviation for the site to inform k and threshold
    site_std = pressure_site["value"].std()

    k = k_multiplier * site_std
    threshold = threshold_multiplier * site_std

    #keep track of both positive and negative cumulative deviations
    positive_cusum = []
    negative_cusum = []

    increased_cumsum = 0
    decreased_cumsum = 0

    for x, mu_t in zip(pressure_site["value"], pressure_site["rolling_mean"]):
        #for the first few readings which won't have a rolling mean
        if pd.isna(mu_t):
            positive_cusum.append(0)
            negative_cusum.append(0)
            continue

        #get the cumulative deviations if pressure keeps going above or below the rolling mean
        increased_cumsum = max(0, increased_cumsum + (x - mu_t - k))
        decreased_cumsum = min(0, decreased_cumsum + (x - mu_t + k))

        positive_cusum.append(increased_cumsum)
        negative_cusum.append(decreased_cumsum)

    pressure_site["cusum_positive"] = positive_cusum
    pressure_site["cusum_negative"] = negative_cusum

    # check if the cumulative deviations have gone above or below the threshold
    pressure_site["pressure_shifts"] = ((pressure_site["cusum_positive"] > threshold) | (pressure_site["cusum_negative"] < -threshold))

    #I'll only consider change events that persist after 2 consecutive readings to filter out noise and reduce false positives
    pressure_site["change_event"] = (pressure_site["pressure_shifts"].rolling(persistence).sum().fillna(0) >= persistence)

    #find deviations from rolling mean
    pressure_site["pressure_deviation"] = pressure_site["value"] - pressure_site["rolling_mean"]
    pressure_site["pressure_deviation_abs"] = (pressure_site["value"] - pressure_site["rolling_mean"]).abs()

    #Because rolling mean is nan for the first few readings, I'll avoid classifying deviations for those readings as rise
    pressure_site["deviation_direction"] = np.where(pressure_site["pressure_deviation"].isna(), None, np.where(pressure_site["pressure_deviation"] < 0, "drop", "rise"))
    
    #get the max absolute cusum value to understand the magnitude of the deviation
    pressure_site["max_abs_cusum"] = pressure_site[["cusum_positive", "cusum_negative"]].abs().max(axis=1)
    pressure_site["site_std"] = site_std
    pressure_site["k"] = k
    pressure_site["threshold"] = threshold

    return pressure_site

#apply the function to get values for all sites
pressure_change_detected = pressure_readings.groupby("site_id").apply(get_cusum_per_site).reset_index(drop=True)
#only retain events when a change event was actually detected 
detections = pressure_change_detected[pressure_change_detected["change_event"]].copy()
detections = detections[["site_id", "dt_stamp", "value", "rolling_mean", "pressure_deviation_abs", "deviation_direction", "cusum_positive", "cusum_negative", "max_abs_cusum"]]

detections.head()


,site_id,dt_stamp,value,rolling_mean,pressure_deviation_abs,deviation_direction,cusum_positive,cusum_negative,max_abs_cusum
342,11,2025-11-24 03:39:00.509280+00:00,54.44,49.950208,4.489792,rise,33.909861,0.0,33.909861
1607,11,2025-12-07 07:54:00.509280+00:00,54.60,51.828229,2.771771,rise,29.758097,0.0,29.758097
1608,11,2025-12-07 08:09:00.509280+00:00,56.09,51.973854,4.116146,rise,30.944531,0.0,30.944531
1609,11,2025-12-07 08:24:00.509280+00:00,58.74,51.957812,6.782188,rise,34.797007,0.0,34.797007
1853,11,2025-12-09 21:24:00.509280+00:00,58.03,50.225521,7.804479,rise,35.394629,0.0,35.394629


### 5. Burst localization


Now that I've detected the abnormal pressure change events, the next step is localization. First, I will enrich the pressure readings with pressure sites data so I can know the site, clusters and pressure thresholds for those sites. I will also identify nearby pipes and sites that may be affected as the pressure readings flow down. I'll also use the site thresholds to define severity levels for the change events. 


In [5]:
#join the detected events with pressure site dataset

site_locations = pressure_sites[["id", "name", "cluster_string", "low_pressure_threshold", "high_pressure_threshold", "geometry"]].rename(columns={"id": "site_id", "name": "site_name"})
detected_sites = detections.merge(site_locations, on="site_id", how="left")
detected_sites.head(3)


#use the pressure thresholds at each site to classify the severity of the detected events
detected_sites["below_low_pressure_threshold"] = (detected_sites["value"] < detected_sites["low_pressure_threshold"])
detected_sites["above_high_pressure_threshold"] = (detected_sites["value"] > detected_sites["high_pressure_threshold"])

def assign_threshold_severity(row):
    if row["below_low_pressure_threshold"] or row["above_high_pressure_threshold"]:
        return "High"
    else:
        return "Medium"

detected_sites["severity"] = detected_sites.apply(assign_threshold_severity, axis=1)

In [6]:
#Next, I'll map each detected site to a pipe segment close to it to establish where the pressure event happened

#First, I'll convert the wkt points in the detections to a metre-based CRS to help get nearest distance in metres
detected_sites_gdf = gpd.GeoDataFrame(detected_sites.copy(),geometry="geometry",crs="EPSG:4326")
detected_sites_mtrs = detected_sites_gdf.to_crs("EPSG:32616")

#for each site, find its location relative to all pipes and get the pipe closest to it and its distance 
nearest_pipe = []
nearest_pipe_distance = []

for site_geom in detected_sites_mtrs["geometry"]:
    distances = pipes_m["geometry"].distance(site_geom)
    nearest_pipe_index = distances.idxmin()
    
    nearest_pipe.append(pipes_m.loc[nearest_pipe_index, "id"])
    nearest_pipe_distance.append(distances.loc[nearest_pipe_index])

#then enrich the detected_sites df
detected_sites["nearest_pipe"] = nearest_pipe
detected_sites["nearest_pipe_distance"] = nearest_pipe_distance

# I only get 1 nearest_pipe_id for each site and all nearest_distance values are 0 which 
# means the pressure site lies directly on a pipe line
#therefore, my location confidence will be high for all detected sites since the site is very close to the pipe segment
def location_confidence(distance):
    if distance <= 200:
        return "High"
    elif distance <= 500:
        return "Medium"
    return "Low"

detected_sites["location_confidence"] = detected_sites["nearest_pipe_distance"].apply(location_confidence)

#I tried to map pipe to an address based on work order records but didnt find any matching pipe ids
detected_sites["nearest_pipe"].isin(work_orders["pipe_id"]).sum()

np.int64(0)

Now that I know the pipes where pressure change may have originated, I need to understand how far the problem extends to find other pipes downstream that may have also be impacted. 

Since the pressure sites have uneven geographic distribution, I should consider the distance to the nearest pressure site when evaluating the significance of a detected event and localizing it. This will also inform my advisory radius for the detected event since the pressure change is likely to propagate down. If the nearest site is close, I can conclude that the event is localized to a smaller radius but if it is farther way, I'll have to increase the radius for the advisory zone.

In [7]:
def nearest_site_distance(row, sites_m):
    site_id = row["site_id"]
    site_geom = row["geometry"]
    other_sites = sites_m[sites_m["site_id"] != site_id].copy()

    #find the distance from the current site to other sites then just keep the closest one
    distances = other_sites.geometry.apply(lambda g: g.distance(site_geom))
    nearest_idx = distances.idxmin()

    return pd.Series({
        "nearest_site_id": other_sites.loc[nearest_idx, "site_id"],
        "nearest_site_distance": distances.loc[nearest_idx]
    })

nearest_site_info = detected_sites_mtrs.apply(
    lambda row: nearest_site_distance(row, detected_sites_mtrs), axis=1)

detected_sites["nearest_site_id"] = nearest_site_info["nearest_site_id"].values
detected_sites["nearest_site_distance"] = nearest_site_info["nearest_site_distance"].values

### 6. Incident Grouping

So far, I know when and where a pressure change happened and its nearest site. I can now group related events into incidents. I will use 30mins as my cut-off which is consistent with the persistence value from earlier and since we're aiming to reduce response time to 30mins. I will also limit the maximum duration of an incident to avoid grouping together events that are too far apart in time and likely unrelated. I chose 12 hours assuming that a crew's shift is 12 hours and they'd want to resolve the incident within their shift. For each incident, I will identify when it started and ended, the sites and clusters affected, deviation type whether increase or drop in pressure, severity level and the distance to the nearest sensor. This incident level summary stats will help me to analyze the impact of each pressure.

In [8]:
detected_sites = detected_sites.sort_values("dt_stamp").copy()

cutoff = pd.Timedelta(minutes=30)
max_incident_duration = pd.Timedelta(hours=12)

incident_ids = []
current_id = 1
incident_start = None
previous_endtime = None

#create a new incident when detections are more than 30 minutes apart or when the current incident has lasted more than 12 hours.
for t in detected_sites["dt_stamp"]:
    if previous_endtime is None:
        incident_start = t
        incident_ids.append(current_id)
    else:
        time_diff = t - previous_endtime
        incident_duration = t - incident_start

        if (time_diff > cutoff) or (incident_duration > max_incident_duration):
            current_id += 1
            incident_start = t

        incident_ids.append(current_id)

    previous_endtime = t

detected_sites["detected_incident_id"] = incident_ids

We will then create a summarized alert feed to help with response and prioritization

In [9]:
incident_summary = (
    detected_sites
    .groupby("detected_incident_id")
    .agg(
        start_time=("dt_stamp", "min"),
        end_time=("dt_stamp", "max"),
        num_detected_incidents=("dt_stamp", "count"),
        site_names=("site_name", lambda x: sorted(set(x.dropna()))),
        num_detected_sites=("site_id", lambda x: len(set(x))),
        clusters_detected=("cluster_string", lambda x: sorted(set(x.dropna()))),
        num_detected_clusters=("cluster_string", lambda x: x.dropna().nunique()),
        max_abs_deviation=("pressure_deviation_abs", "max"),
        max_abs_cusum=("max_abs_cusum", "max"),
        drop_count=("deviation_direction", lambda x: (x == "drop").sum()),
        rise_count=("deviation_direction", lambda x: (x == "rise").sum()),
        nearest_pipe_ids=("nearest_pipe", lambda x: sorted(set(x.dropna().astype(int)))),
        nearest_site_ids=("nearest_site_id", lambda x: sorted(set(x.dropna().astype(int)))),
        nearest_site_distance=("nearest_site_distance", "min"),
        location_confidence=("location_confidence", lambda x: "High" if "High" in set(x) else "Medium" if "Medium" in set(x) else "Low"),
        severity=("severity", lambda x: "High" if "High" in set(x) else "Medium" if "Medium" in set(x) else "Low")
    )
    .reset_index()
)

In [10]:
#I can enrich this feed further by calculating duration and dominant direction of pressure change for each incident

incident_summary["duration_in_minutes"] = (incident_summary["end_time"] - incident_summary["start_time"]).dt.total_seconds() / 60

incident_summary["dominant_direction"] = np.where(incident_summary["drop_count"] >= incident_summary["rise_count"],"drop", "rise")

incident_summary["site_distance_percentile"] = (incident_summary["nearest_site_distance"].rank(pct=True))

incident_summary["pressure_drop_ratio"] = (incident_summary["drop_count"] /(incident_summary["drop_count"] + incident_summary["rise_count"]))

### 7. Advisory Zone Definition


In this section, I will define the advisory zone radius based on distance to the nearest sensor, strength of pressure drop, and the number of clusters and sites impacted based on the grouped feed. I will have a wider advisory zone for areas that are farther from its nearest site, or where I see multiple clusters being affected by a failure, or where the failure is of a severe type like pressure issue/main break, etc causing a significant drop in the site. 

For areas near multiple sensors, it is easier to identify the potentially affected pipes using the network of sensors so I will maintain a smaller radius. To avoid over-extending the radius and introducing lots of false positives for areas that lack enough sensor converage, I will cap the radius. In this case, I will cap it to 4000m, a few metres shy of the max distance (4403m) to a next site. 

In [11]:
incident_summary['nearest_site_distance'].describe()

def dynamic_radius(row):

    #I'll start with a small base radius to account for location uncertainty
    radius = 100

    # Sensor spacing
    #this should be the most important factor in determining the radius as it gives me a sense of how much the incident
    #is localized based on how closely spaced the sites are in that area. I'll use a smaller radius if the nearest other site is very close

    if row["site_distance_percentile"] > 0.75: 
        radius += 3000 #I estiamted it based on the median nearest other site distance of around 2.3km with a std of 735 for the top 25% most spaced out sites
    elif row["site_distance_percentile"] > 0.5:
        radius += 2000 #I estimated from the mean-std dev
    elif row["site_distance_percentile"] > 0.25:
        radius += 1200  #this was also an estimate from min -std dev, as areas with more sensors mean the change can be detected more easily by nearby sensors

    #increase the radius if incident occurs across several sites and clusters
    #i gave more weight to multiple sites than clusters because that covers more radius as we can also have multiple clusters in a site which may make them closer    
    if row["num_detected_sites"] >= 2:
        radius += 750 
    if row["num_detected_clusters"] >= 2:
        radius += 500

    #increase radius when pressure drop is high ->proxy for pressure, main break events 
    #I kept the radius adjustments smaller since this is not very consequential like the nearest site distance
    if row["pressure_drop_ratio"] >= 0.75:
        radius += 800
    elif row["pressure_drop_ratio"] >= 0.45:
        radius += 600
    elif row["pressure_drop_ratio"] >= 0.20:
        radius += 250

    #cap the radius - to about max radius
    return min(radius, 4000)

incident_summary["radius"] = incident_summary.apply(dynamic_radius, axis=1)

#we'll now use this radius to find potentially affected pipes for each incident

In [12]:
#first, I'll update detected_sites_mtrs with the detected_incident_id so I can use it to find candidate pipes for each incident based on the dynamic radius calculated for that incident

detected_sites_gdf = gpd.GeoDataFrame(detected_sites.copy(),geometry="geometry", crs="EPSG:4326")
detected_sites_mtrs = detected_sites_gdf.to_crs("EPSG:32616")

def get_candidate_pipes(incident_id, radius, detected_sites_mtrs, pipes_m):
    incident_detected = detected_sites_mtrs[detected_sites_mtrs["detected_incident_id"] == incident_id]
    candidate_pipe_ids = set()

    #for each incident detected, find all pipes within its radius
    for detection in incident_detected["geometry"]:
        distances = pipes_m["geometry"].distance(detection)
        nearby_pipes = pipes_m.loc[distances <= radius, "id"]
        candidate_pipe_ids.update(nearby_pipes.dropna().astype(str).tolist())
    return sorted(candidate_pipe_ids)


incident_summary["candidate_pipe_ids"] = incident_summary.apply(
    lambda row: get_candidate_pipes(row["detected_incident_id"], row["radius"], detected_sites_mtrs, pipes_m),axis=1)

incident_summary["num_candidate_pipes"] = incident_summary["candidate_pipe_ids"].apply(len)

In [13]:
#incident feed report

incident_feed = incident_summary[["detected_incident_id", "start_time", "end_time", "severity", "location_confidence", "duration_in_minutes", "radius", "num_detected_sites", "num_detected_clusters", "num_candidate_pipes"]]

incident_feed_display = incident_feed.rename(columns={
    "detected_incident_id": "Incident ID",
    "start_time": "Start Time",
    "end_time": "End Time",
    "duration_in_minutes": "Duration (mins)",
    "severity": "Severity",
    "location_confidence": "Location Confidence",
    "radius": "Advisory Radius (m)",
    "num_candidate_pipes": "Number of Candidate Pipes",
    "num_detected_sites": "No. Affected Sites",
    "num_detected_clusters": "No. Affected Clusters",
})

incident_feed_display.to_html("cusum_incident_feed.html", index=False, classes="alert-table", border=0)

### 8. Validation against historical events

In [14]:
#merge the validation sets into an enriched wide format for easier analysis of validation results
event_validation = event_affected_mains.merge(
    event_alerts[["id", "name", "from_date", "to_date", "city_id"]],
    left_on="event_alert_id",
    right_on="id",
    how="left",
    suffixes=("_main", "_alert")
)

event_validation = event_validation.rename(columns={"water_main_id": "pipe_id"})

In [15]:
cusum_validation_rows = []

for event_id, event_group in event_validation.groupby("event_alert_id"):

    event_name = event_group["name"].iloc[0]
    start_time = event_group["from_date"].iloc[0]
    end_time = event_group["to_date"].iloc[0]

    #for each event, get all pipes from the validation set
    actual_pipes = set(event_group["pipe_id"].dropna().astype(int))

    #find detected incidents that appear in the the validation set
    detected_incidents = incident_summary[(incident_summary["start_time"] <= end_time) & (incident_summary["end_time"] >= start_time)]
    count_detected = len(detected_incidents) > 0

    predicted_pipes = set()

    for pipes_list in detected_incidents["candidate_pipe_ids"]:
        if isinstance(pipes_list, list):
            predicted_pipes.update([int(p) for p in pipes_list])

    true_positives = predicted_pipes & actual_pipes
    false_positives = predicted_pipes - actual_pipes
    false_negatives = actual_pipes - predicted_pipes

    cusum_validation_rows.append({
        "event_alert_id": event_id,
        "event_name": event_name,
        "event_start_time": start_time,
        "event_end_time": end_time,
        "num_actual_pipes": len(actual_pipes),
        "num_predicted_pipes": len(predicted_pipes),
        "true_positive_pipes": len(true_positives),
        "false_positive_pipes": len(false_positives),
        "false_negative_pipes": len(false_negatives),
        "precision": (len(true_positives) / len(predicted_pipes) if predicted_pipes else 0)*100,
        "recall": (len(true_positives) / len(actual_pipes) if actual_pipes else 0)*100,
    })

cusum_validation_results = pd.DataFrame(cusum_validation_rows)

cusum_validation_results.to_html("cusum_validation_results.html", index=False, classes="alert-table", border=0)

cusum_validation_results

,event_alert_id,event_name,event_start_time,event_end_time,num_actual_pipes,num_predicted_pipes,true_positive_pipes,false_positive_pipes,false_negative_pipes,precision,recall
0,5,Main Break - Oak Street,2026-01-05 14:09:00.509280+00:00,2026-01-06 06:09:00.509280+00:00,1105,553,522,31,583,94.394213,47.239819
1,6,Pressure Loss - Industrial Zone,2026-02-02 14:09:00.509280+00:00,2026-02-03 01:09:00.509280+00:00,27,218,27,191,0,12.385321,100.000000
2,7,Valve Failure - Downtown,2026-01-12 14:09:00.509280+00:00,2026-01-12 21:09:00.509280+00:00,3131,1872,1794,78,1337,95.833333,57.297988
3,8,Pump Station Outage - North,2026-01-23 14:09:00.509280+00:00,2026-01-24 03:09:00.509280+00:00,1105,553,522,31,583,94.394213,47.239819


### DBSCAN implementation from customer complaints

A single customer complaint may not help us detect and find a burst. However, if several complaints happen within the same area and in the same timeframe, we may be able to identify patterns by grouping the complaints together. 

### 9. Create complaint clusters using DBSCAN

In [16]:
#let's check the number of complaints in 12 hour windows to see the typical complaint volume 
#so that we can have a better idea of what to set the min_samples parameter to.

window_counts = (
    complaints
    .groupby(pd.Grouper(key="recorded_at", freq="12H"))
    .size()
    .reset_index(name="num_complaints")
)

window_counts["num_complaints"].describe()

#since the avg complaints is about 8 with a std dev of 3, we can experiment with min_samples of about 3-11 so we dont
# end up excluding events when we didn't get enough complaints to form a cluster. I experimented with a few values and ended up 
# going with 3 as I got a better result

count    361.000000
mean       8.437673
std        2.815062
min        1.000000
25%        7.000000
50%        8.000000
75%       10.000000
max       16.000000
Name: num_complaints, dtype: float64

In [17]:
 #I'll group complaints into 12 hour time buckets and run DBSCAN on each bucket to find clusters of complaints that are close in space and time.

def run_dbscan_per_timebucket(timebucket_df, eps=0.03, min_samples=3):

    timebucket_df = timebucket_df.copy()

    #if number of complaints in a time bucket is less than min_samples, I'll assign that bucket to the noise cluster
    if len(timebucket_df) < min_samples:
        timebucket_df["complaint_cluster"] = -1
        return timebucket_df

    coordinates = timebucket_df[["longitude", "latitude"]].values

    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(coordinates)

    timebucket_df["complaint_cluster"] = labels
    return timebucket_df


complaints_m = complaints_m.sort_values("recorded_at").copy()
bucket_size = "12H"
clustered_buckets = []

for start_time, timebucket_df in complaints_m.groupby(pd.Grouper(key="recorded_at", freq=bucket_size)):
    result = run_dbscan_per_timebucket(timebucket_df)
    result["bucket_start"] = start_time
    result["bucket_end"] = start_time + pd.Timedelta(bucket_size)
    clustered_buckets.append(result)

complaint_clusters = pd.concat(clustered_buckets, ignore_index=True)

#I'll ignore the noise points with cluster= -1 and focus only on the valid clusters
valid_clusters = complaint_clusters[complaint_clusters["complaint_cluster"] != -1].copy()

valid_clusters.head(3)

,id,request_id,complaint_theme,priority,recorded_at,geom_wkt,geometry,longitude,latitude,complaint_cluster,bucket_start,bucket_end
0,3224,REQ-10035,Missing/Loose - Cover/Screw,Medium,2025-08-22 16:08:59.951581+00:00,POINT(-89.65091795634741 39.77594122562612),POINT (272965.738 4406250.928),-89.650918,39.775941,0,2025-08-22 12:00:00+00:00,2025-08-23 00:00:00+00:00
1,5111,REQ-11922,Other,Medium,2025-08-22 16:08:59.951581+00:00,POINT(-89.66776949999999 39.7944515),POINT (271583.587 4408348.612),-89.667769,39.794452,0,2025-08-22 12:00:00+00:00,2025-08-23 00:00:00+00:00
2,3585,REQ-10396,No Water,Low,2025-08-22 17:08:59.951581+00:00,POINT(-89.67045945000002 39.81293955000001),POINT (271414.521 4410407.814),-89.670459,39.812940,0,2025-08-22 12:00:00+00:00,2025-08-23 00:00:00+00:00


### 10. Create a complaints incident summary feed for burst detection

Now that I have the complaint clusters, I can group them into incidents based on the start time and cluster it was assigned to. Then, I can analyze each incident and see the number of complaints, the priorities and themes present as well as the centre of the cluster.   

In [18]:
valid_clusters["dbscan_incident_id"] = (valid_clusters["bucket_start"].astype(str)+ "_" + valid_clusters["complaint_cluster"].astype(str))

dbscan_summary = (
    valid_clusters
    .groupby(["bucket_start", "bucket_end", "complaint_cluster","dbscan_incident_id"])
    .agg(
        start_time=("recorded_at", "min"),
        end_time=("recorded_at", "max"),
        num_complaints=("recorded_at", "count"),
        centroid_lon=("longitude", "mean"),
        centroid_lat=("latitude", "mean"),
        all_priorities=("priority", lambda x: x.dropna().value_counts().to_dict()),
        all_complaints=("complaint_theme", lambda x: x.dropna().value_counts().to_dict()),
    )
    .reset_index()
)

dbscan_summary.head(3)

,bucket_start,bucket_end,complaint_cluster,dbscan_incident_id,start_time,end_time,num_complaints,centroid_lon,centroid_lat,all_priorities,all_complaints
0,2025-08-22 12:00:00+00:00,2025-08-23 00:00:00+00:00,0,2025-08-22 12:00:00+00:00_0,2025-08-22 16:08:59.951581+00:00,2025-08-22 23:08:59.951581+00:00,5,-89.657707,39.789610,"{'Low': 3, 'Medium': 2}","{'Missing/Loose - Cover/Screw': 1, 'Other': 1,..."
1,2025-08-23 00:00:00+00:00,2025-08-23 12:00:00+00:00,0,2025-08-23 00:00:00+00:00_0,2025-08-23 02:08:59.951581+00:00,2025-08-23 10:08:59.951581+00:00,8,-89.672132,39.781896,"{'High': 4, 'Low': 2, 'Medium': 2}","{'Pressure Problem': 4, 'Water in Building': 1..."
2,2025-08-23 12:00:00+00:00,2025-08-24 00:00:00+00:00,0,2025-08-23 12:00:00+00:00_0,2025-08-23 16:08:59.951581+00:00,2025-08-23 22:08:59.951581+00:00,4,-89.685095,39.797331,"{'High': 2, 'Low': 1, 'Medium': 1}","{'Water in Building': 1, 'No Water': 1, 'Press..."


### 11. Advisory Zone Definition

I will use the comprehensive complaint themes and priorities in the cluster to rank the complaints and priorities in the cluster. I'll also use these together with the number of complaints per cluster to create the dynamic radius for boil water advisory zone definition for the identified burst clusters.

In [19]:
#I'll create a weighted severity score for each cluster based on the complaint themes in that cluster

def theme_severity(themes):
    #I'll assign higher scores to main issues like no water or pressure problems and lower scores to things that may not be pressure related
    theme_score = {
        "No Water": 5,
        "Pressure Problem": 5,
        "Water Coming Up": 4,
        "Water in Building": 2,
        "Missing/Loose - Cover/Screw": 1,
        "Other": 0.5
    }

    #I'll give a small score to complaints that fall outside the above in case the data changes later
    if not isinstance(themes, dict) or len(themes) == 0:
        return 0.5

    total_score = 0

    for theme, count in themes.items():
        score = theme_score.get(theme, 0.5)
        total_score += score * count

    return total_score


dbscan_summary["theme_severity_score"] = dbscan_summary["all_complaints"].apply(theme_severity)

#I'll do the same for priority
def priority_scoring(priority):
    #I'll use a scoring of 1-3- 3 for high priority complaints
    priority_score = {
        "High": 3,
        "Medium": 2,
        "Low": 1
    }

    #I'll give a small score to priorities that fall outside the above in case the data changes later
    if not isinstance(priority, dict) or len(priority) == 0:
        return 0.5

    total_score = 0

    for p, count in priority.items():
        score = priority_score.get(p, 0.5)
        total_score += score * count

    return total_score


dbscan_summary["priority_score"] = dbscan_summary["all_priorities"].apply(priority_scoring)

dbscan_summary.head(3)

,bucket_start,bucket_end,complaint_cluster,dbscan_incident_id,start_time,end_time,num_complaints,centroid_lon,centroid_lat,all_priorities,all_complaints,theme_severity_score,priority_score
0,2025-08-22 12:00:00+00:00,2025-08-23 00:00:00+00:00,0,2025-08-22 12:00:00+00:00_0,2025-08-22 16:08:59.951581+00:00,2025-08-22 23:08:59.951581+00:00,5,-89.657707,39.789610,"{'Low': 3, 'Medium': 2}","{'Missing/Loose - Cover/Screw': 1, 'Other': 1,...",15.5,7
1,2025-08-23 00:00:00+00:00,2025-08-23 12:00:00+00:00,0,2025-08-23 00:00:00+00:00_0,2025-08-23 02:08:59.951581+00:00,2025-08-23 10:08:59.951581+00:00,8,-89.672132,39.781896,"{'High': 4, 'Low': 2, 'Medium': 2}","{'Pressure Problem': 4, 'Water in Building': 1...",31.5,18
2,2025-08-23 12:00:00+00:00,2025-08-24 00:00:00+00:00,0,2025-08-23 12:00:00+00:00_0,2025-08-23 16:08:59.951581+00:00,2025-08-23 22:08:59.951581+00:00,4,-89.685095,39.797331,"{'High': 2, 'Low': 1, 'Medium': 1}","{'Water in Building': 1, 'No Water': 1, 'Press...",13.0,9


In [20]:
#I'll use the complaint theme score and priority assigned to help determine the radius that may be affected
def dbscan_dynamic_radius(row):

    #i'll start with a small radius around the cluster to account for the initial uncertainty
    radius = 100

    theme_score = row["theme_severity_score"]
    priority_score = row["priority_score"]

    #complaint themes are more likely to help isolate real issues and can make assessment easier while priority assigned could be more subjective
    total_score= (theme_score * 0.9) + (priority_score *0.1)

    #since highest theme score is 5 and highest priority is 3, that means just 1 compliant in that cluster would get 4.8
    #and since our cluster has a min of 3 compliants, that means a small but serious cluster can get to about 14 hence why 
    #I set the max threshold to 15, then reduced the radius for the cluseters with lower scores. A severe issue may require wider radius to be looked at. 

    if total_score >= 15:
        radius += 2000
    elif total_score >= 10:
        radius += 1500
    elif total_score >= 8:
        radius += 800
    elif total_score >= 5:
        radius += 400
    else:
        radius += 0
 

    return min(radius, 2500)

dbscan_summary["cluster_radius"] = dbscan_summary.apply(dbscan_dynamic_radius, axis=1)

In [21]:
#The complaint theme severity score, number of complaints in the cluster and the cluster radius can give us an idea of how confident we are about the location of the issue. 
#more complaints, higher severity score and smaller cluster radius gives us more confidence about the location of the issue. 
#So I'll these factors to classify the location confidence of each cluster as high, medium or low.

#I'll use the summary stats of these columns to set some thresholds for the classification

dbscan_summary["theme_severity_score"].describe()
dbscan_summary["cluster_radius"].describe()
dbscan_summary["num_complaints"].describe()

def dbscan_location_confidence(row):
    #high confidence- high num_complaints and theme_severity_score and small cluster_radius, 
    #used mean num_complaints and mean theme_severity_score; used 25% percentile for the cluster radius
    if (row["num_complaints"] >= 6 and row["theme_severity_score"] >= 15 and row["cluster_radius"] <= 900):
        return "High"
    
    #kept lowest no. of complaints to form a cluster as medium; used mean -std dev for theme_severity_score; used 50% percentile for the cluster radius
    elif (row["num_complaints"] >= 3 and row["theme_severity_score"] >= 7 and row["cluster_radius"] <= 1600):
        return "Medium"
    else:
        return "Low"

dbscan_summary["location_confidence"] = dbscan_summary.apply(dbscan_location_confidence, axis=1)

### 12. Localization of complaints

To correctly localize the complaints, we'll need to identify all pipes that fall within each cluster's advisory radius. We'll use the cluster radius calculated above to create a buffer around each complaint.

In [22]:
#we'll find the cluster radius for each incident and enrich the valid_clusters dataframe with the cluster radius
cluster_radii = dbscan_summary[["dbscan_incident_id", "cluster_radius"]].copy()


valid_clusters = valid_clusters.copy()
valid_clusters["geometry"] = valid_clusters["geom_wkt"].apply(wkt.loads)

valid_clusters_gdf = gpd.GeoDataFrame(valid_clusters, geometry="geometry",crs="EPSG:4326")
valid_clusters_mtrs = valid_clusters_gdf.to_crs("EPSG:32616")
valid_clusters_and_their_radius = valid_clusters_mtrs.merge(cluster_radii, on="dbscan_incident_id", how="left")

#create an advisory zone around each complaint point
complaint_zone = valid_clusters_and_their_radius.copy()
complaint_zone["zone_geometry"] = complaint_zone.geometry.buffer(complaint_zone["cluster_radius"])

#use the advisory zone to find pipes that intersect with them to get candidate pipes for each cluster
complaint_buffers_gdf = gpd.GeoDataFrame(complaint_zone, geometry="zone_geometry", crs= "EPSG:32616")

intersecting_pipes = gpd.sjoin(
    complaint_buffers_gdf[["dbscan_incident_id", "zone_geometry"]],
    pipes_m[["id", "geometry"]], how="left", predicate="intersects")


#we'll aggregate the intersecting pipes to get final list of candidate pipes per incident
candidate_pipes_by_cluster = (
    intersecting_pipes
    .groupby("dbscan_incident_id")["id"]
    .apply(lambda x: sorted(set(x.dropna().astype(int))))
    .reset_index()
    .rename(columns={"id": "candidate_pipe_ids"})
)

candidate_pipes_by_cluster["num_candidate_pipes"] = (candidate_pipes_by_cluster["candidate_pipe_ids"].apply(len))

dbscan_summary = dbscan_summary.merge(candidate_pipes_by_cluster, on="dbscan_incident_id", how="left")

dbscan_summary["num_candidate_pipes"] = dbscan_summary["candidate_pipe_ids"].apply(len)

In [23]:
dbscan_incident_feed = dbscan_summary[['dbscan_incident_id','start_time','end_time','location_confidence','num_complaints', 'all_complaints', 'all_priorities','theme_severity_score', 'priority_score','cluster_radius','num_candidate_pipes']]
dbscan_incident_feed.to_html("dbscan_incident_feed.html", index=False, classes="alert-table", border=0)

dbscan_incident_feed.head(3)


,dbscan_incident_id,start_time,end_time,location_confidence,num_complaints,all_complaints,all_priorities,theme_severity_score,priority_score,cluster_radius,num_candidate_pipes
0,2025-08-22 12:00:00+00:00_0,2025-08-22 16:08:59.951581+00:00,2025-08-22 23:08:59.951581+00:00,Medium,5,"{'Missing/Loose - Cover/Screw': 1, 'Other': 1,...","{'Low': 3, 'Medium': 2}",15.5,7,1600,2766
1,2025-08-23 00:00:00+00:00_0,2025-08-23 02:08:59.951581+00:00,2025-08-23 10:08:59.951581+00:00,Low,8,"{'Pressure Problem': 4, 'Water in Building': 1...","{'High': 4, 'Low': 2, 'Medium': 2}",31.5,18,2100,4931
2,2025-08-23 12:00:00+00:00_0,2025-08-23 16:08:59.951581+00:00,2025-08-23 22:08:59.951581+00:00,Medium,4,"{'Water in Building': 1, 'No Water': 1, 'Press...","{'High': 2, 'Low': 1, 'Medium': 1}",13.0,9,1600,1995


### 13. Validation against historical events

In [24]:
dbscan_results = []

for event_id, event_group in event_validation.groupby("event_alert_id"):

    event_name = event_group["name"].iloc[0]
    event_start = event_group["from_date"].iloc[0]
    event_end = event_group["to_date"].iloc[0]

    actual_pipes = set(event_group["pipe_id"].dropna().astype(int))

    # DBSCAN clusters that appear in the validation set
    predicted_clusters = dbscan_summary[(dbscan_summary["start_time"] <= event_end) & (dbscan_summary["end_time"] >= event_start)]
    
    predicted_pipes = set()

    for pipe_list in predicted_clusters["candidate_pipe_ids"]:
        if isinstance(pipe_list, list):
            predicted_pipes.update([int(p) for p in pipe_list])

    true_positives = predicted_pipes & actual_pipes
    false_positives = predicted_pipes - actual_pipes
    false_negatives = actual_pipes - predicted_pipes

    dbscan_results.append({
        "event_alert_id": event_id,
        "event_name": event_name,
        "event_start_time": event_start,
        "event_end_time": event_end,
        "num_complaints_in_clusters": predicted_clusters["num_complaints"].sum(),
        "num_actual_pipes": len(actual_pipes),
        "num_predicted_pipes": len(predicted_pipes),
        "true_positive_pipes": len(true_positives),
        "false_positive_pipes": len(false_positives),
        "false_negative_pipes": len(false_negatives),
        "precision": (len(true_positives) / len(predicted_pipes) if predicted_pipes else 0)*100,
        "recall": (len(true_positives) / len(actual_pipes) if actual_pipes else 0)*100,
    })

dbscan_validation_results = pd.DataFrame(dbscan_results)

dbscan_validation_results.to_html("dbscan_validation_results.html", index=False, classes="alert-table", border=0)

dbscan_validation_results

,event_alert_id,event_name,event_start_time,event_end_time,num_complaints_in_clusters,num_actual_pipes,num_predicted_pipes,true_positive_pipes,false_positive_pipes,false_negative_pipes,precision,recall
0,5,Main Break - Oak Street,2026-01-05 14:09:00.509280+00:00,2026-01-06 06:09:00.509280+00:00,9,1105,2459,0,2459,1105,0.000000,0.000000
1,6,Pressure Loss - Industrial Zone,2026-02-02 14:09:00.509280+00:00,2026-02-03 01:09:00.509280+00:00,4,27,21,0,21,27,0.000000,0.000000
2,7,Valve Failure - Downtown,2026-01-12 14:09:00.509280+00:00,2026-01-12 21:09:00.509280+00:00,7,3131,3256,1436,1820,1695,44.103194,45.863941
3,8,Pump Station Outage - North,2026-01-23 14:09:00.509280+00:00,2026-01-24 03:09:00.509280+00:00,12,1105,5402,935,4467,170,17.308404,84.615385
